## Source

In [1]:
import sys
sys.path.append('/root/ros_ws/devel/lib/python3/dist-packages')
sys.path.append('/opt/ros/noetic/lib/python3/dist-packages')

## Imports

In [2]:
import rospy
import actionlib
import threading
import ipywidgets as widgets
from IPython.display import clear_output
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
from assignment_2_2024.msg import RobotOdom, PlanningAction, PlanningGoal
from actionlib_msgs.msg import GoalStatus

## Set or cancel a target

In [9]:
def set_target_client():
    client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
    client.wait_for_server()
    
    connected = client.wait_for_server(timeout=rospy.Duration(5.0))
    if not connected:
        rospy.logerr("Could not connect to /reaching_goal action server.")
    else:
        rospy.loginfo("Connected to /reaching_goal action server.")

    # Widgets
    bt_send = widgets.Button(description="Set New Target", style={'button_color':'yellow'})
    display(bt_send)

    def on_click_new(b):
        clear_output()
        print("Select a new position target:")
        x_input = widgets.FloatText(
            value=0.0,
            description='X target:'
        )

        y_input = widgets.FloatText(
            value=0.0,
            description='Y target:'
        )
        
        display(x_input, y_input)
     
        bt_confirm = widgets.Button(description="Confirm New Target", style={'button_color':'lightgreen'})
        display(bt_confirm)
        def on_click_confirm(b):
            x_tar = x_input.value
            y_tar = y_input.value
            rospy.set_param('/des_pos_x', x_tar)
            rospy.set_param('/des_pos_y', y_tar)
            goal = PlanningGoal()
            goal.target_pose.pose.position.x = x_tar
            goal.target_pose.pose.position.y = y_tar
            client.send_goal(goal)
            clear_output()
            rospy.loginfo(f"New target has been set : x={goal.target_pose.pose.position.x}, y={goal.target_pose.pose.position.y}")
            rospy.loginfo("Going to target...")
            
            bt_cancel = widgets.Button(description="Cancel Target", style={'button_color':'red'})
            display(bt_cancel)
            
            def reset_ihm(b):
                    clear_output()
                    bt_send = widgets.Button(description="Set New Target", style={'button_color':'yellow'})
                    display(bt_send)
                    bt_send.on_click(on_click_new)
            
            def on_click_cancel(b):
                clear_output()
                rospy.loginfo("Trying to cancel current target...")
                
                if client.get_state() == GoalStatus.ACTIVE:
                    rospy.loginfo(f"Cancelling target (x={goal.target_pose.pose.position.x}, y={goal.target_pose.pose.position.y})...")
                    client.cancel_goal()
                    rospy.loginfo("Target cancelled.")
                    bt_C = False
                else:
                    print(client.get_state())
                    rospy.loginfo("The robot is not currently aiming at a target.")
                
                bt_clear = widgets.Button(description="OK", style={'button_color':'lightblue'})
                display(bt_clear)
                bt_clear.on_click(reset_ihm)
            bt_cancel.on_click(on_click_cancel)
            
            def check_success():
                client.wait_for_result()
                print(client.get_state())
                if client.get_state() == GoalStatus.SUCCEEDED:
                    clear_output()
                    rospy.loginfo(f"Target (x={goal.target_pose.pose.position.x}, y={goal.target_pose.pose.position.y}) reached.")
                    bt_clear = widgets.Button(description="Success. Click here", style={'button_color':'lightblue'})
                    display(bt_clear)
                    bt_clear.on_click(reset_ihm)
            threading.Thread(target=check_success).start()
        bt_confirm.on_click(on_click_confirm)
    bt_send.on_click(on_click_new)

## Position and distance to the obstacles feedback

In [10]:
def get_feedback():
    # Widgets
    x_widget = widgets.FloatText(description='X:', disabled=True)
    y_widget = widgets.FloatText(description='Y:', disabled=True)
    min_dist_widget = widgets.FloatText(description='Obstacle [m]:', disabled=True)
    pos_box = widgets.VBox([widgets.Label("Robot Position:"), x_widget, y_widget])
    laser_box = widgets.VBox([widgets.Label("Closest Obstacle:"), min_dist_widget])
    main_box = widgets.HBox([pos_box, laser_box])
    display(main_box)

    # Callbacks
    def pos_callback(msg):
        x_widget.value = msg.pose.pose.position.x
        y_widget.value = msg.pose.pose.position.y

    def laser_callback(msg):
        min_dist_widget.value = min(msg.ranges)

    # Subscribers /odom, /scan
    rospy.Subscriber('/odom', Odometry, pos_callback)
    rospy.Subscriber('/scan', LaserScan, laser_callback)

## Display the interface

In [11]:
if not rospy.core.is_initialized():
    rospy.init_node('action_client', anonymous=True)

In [12]:
get_feedback()

In [13]:
set_target_client()

Button(description='Set New Target', style=ButtonStyle(button_color='yellow'))